# Blinded human evaluation

Human reviewers see anonymous responses A and B. They work independently and cannot see the private mapping until every review is complete. Use `scripts/prepare_human_review.py` to create sheets and `scripts/score_human_review.py` to unblind them. This notebook only displays the finished results.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Change this to the folder passed to score_human_review.py.
HUMAN_EVAL_DIR = Path(os.environ.get('MFR_HUMAN_EVAL_DIR', 'human_eval_results')).expanduser().resolve()
print(HUMAN_EVAL_DIR)

## Preference result
A score above 50% favors the candidate. The confidence interval is bootstrapped across prompts so multiple reviewers do not artificially increase the sample size.

In [ ]:
summary_path = HUMAN_EVAL_DIR / 'human_summary.csv'
if summary_path.exists():
    summary = pd.read_csv(summary_path)
    display(summary.round(2))
    chart = summary.set_index('behavior')
    errors = [chart['candidate_score_pct'] - chart['ci_low_pct'], chart['ci_high_pct'] - chart['candidate_score_pct']]
    chart['candidate_score_pct'].plot.bar(yerr=errors, capsize=4, figsize=(8, 4), ylim=(0, 100))
    plt.axhline(50, color='black', linestyle='--', linewidth=1)
    plt.ylabel('Candidate preference score (%)'); plt.title('Blinded human preference')
    plt.xticks(rotation=0); plt.tight_layout(); plt.show()
else: print('Run score_human_review.py first:', summary_path)

## Reviewer agreement
Agreement is supporting evidence about annotation consistency; it is not a model-quality score.

In [ ]:
agreement_path = HUMAN_EVAL_DIR / 'reviewer_agreement.csv'
display(pd.read_csv(agreement_path).round(2)) if agreement_path.exists() else print('No agreement file found.')